# Machine Learning: From Features to Cost-Based Decisions

Notebook for **Chapter 6** of the *ways* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ways/ch06.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
# Setup: the files used in this chapter are downloaded next to the notebook.
import os, urllib.request
os.makedirs("data", exist_ok=True)
for f in ["customers.csv", "orders.csv", "products.csv", "shopkart.db"]:
    if not os.path.exists("data/" + f):
        urllib.request.urlretrieve("https://dadipavan.github.io/Learning/study/ways/data/" + f, "data/" + f)


## 6.1 Frame the prediction problem

Five questions define any ML problem. Answer them in writing before touching a model.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep, mlprep

o, c, p = prep.load_clean(DATA)
d = mlprep.late_frame(o)
train = d[d.order_date < "2024-10-01"]
test = d[d.order_date >= "2024-10-01"]
print("orders with a target:", len(d), "| late share:", round(d.late.mean(), 4))
print("train (Jan-Sep):", len(train), round(train.late.mean(), 4), "| test (Oct-Dec):", len(test), round(test.late.mean(), 4))
print("accuracy of always saying on time:", round(1 - test.late.mean(), 4))
print(d.groupby("promised_minutes").late.mean().round(4).to_dict())

orders with a target: 17208 | late share: 0.1376
train (Jan-Sep): 12966 0.1361 | test (Oct-Dec): 4242 0.1419
accuracy of always saying on time: 0.8581
{45.0: 0.2499, 60.0: 0.028}


## 6.2 Features and leakage

**Feature engineering** turns raw columns into inputs a model can use: category columns become one-hot columns (one 0/1 column per value), numbers are scaled for models that care (logistic regression), and we add knowledge-based features such as `evening` (hours 18 to 21) and `weekend`. Tree models can find such thresholds themselves; linear models cannot, so a feature you add by hand is a gift to them.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep, mlprep
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

o, c, p = prep.load_clean(DATA)
d = mlprep.late_frame(o)
train, test = d[d.order_date < "2024-10-01"], d[d.order_date >= "2024-10-01"]

def fit_auc(num):
    pre = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), mlprep.CAT), ("n", StandardScaler(), num)])
    m = make_pipeline(pre, HistGradientBoostingClassifier(random_state=0)).fit(train[mlprep.CAT + num], train.late)
    return roc_auc_score(test.late, m.predict_proba(test[mlprep.CAT + num])[:, 1])

print("honest features      AUC:", round(fit_auc(mlprep.NUM), 4))
print("with delivery_minutes AUC:", round(fit_auc(mlprep.NUM + ["delivery_minutes"]), 4))

honest features      AUC: 0.8363
with delivery_minutes AUC: 1.0


In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep, mlprep
from sklearn.metrics import roc_auc_score

o, c, p = prep.load_clean(DATA)
d = mlprep.late_frame(o)
train, test = d[d.order_date < "2024-10-01"], d[d.order_date >= "2024-10-01"]

# leaky: customer's mean lateness over ALL their orders, including the test order itself
leaky = d.groupby("customer_id").late.transform("mean")[test.index]
print("leaky customer-rate feature AUC on test:", round(roc_auc_score(test.late, leaky), 4))

# honest: only orders before October
hist = train.groupby("customer_id").late.mean()
honest = test.customer_id.map(hist).fillna(train.late.mean())
print("honest customer-rate feature AUC on test:", round(roc_auc_score(test.late, honest), 4))

leaky customer-rate feature AUC on test: 0.8553
honest customer-rate feature AUC on test: 0.4953


## 6.3 Validation done right

A model must be judged on data it has not seen, and in a way that mimics its future use. The usual recipe is a split: **train** to fit, **validation** to choose settings, **test** touched once at the end. Three styles exist.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep, mlprep
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import roc_auc_score

o, c, p = prep.load_clean(DATA)
d = mlprep.late_frame(o).sort_values("order_date").reset_index(drop=True)
X, y = d[mlprep.CAT + mlprep.NUM], d.late
pre = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), mlprep.CAT), ("n", StandardScaler(), mlprep.NUM)])
model = make_pipeline(pre, LogisticRegression(max_iter=1000))

Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, random_state=0)
print("random split AUC :", round(roc_auc_score(yb, model.fit(Xa, ya).predict_proba(Xb)[:, 1]), 4))

cut = int(len(d) * 0.75)
model.fit(X.iloc[:cut], y.iloc[:cut])
print("time split AUC   :", round(roc_auc_score(y.iloc[cut:], model.predict_proba(X.iloc[cut:])[:, 1]), 4))

scores = []
for tr, te in TimeSeriesSplit(n_splits=5).split(X):
    model.fit(X.iloc[tr], y.iloc[tr])
    scores.append(roc_auc_score(y.iloc[te], model.predict_proba(X.iloc[te])[:, 1]))
print("5 forward-chaining folds:", np.round(scores, 4), "mean", round(np.mean(scores), 4))

random split AUC : 0.8324
time split AUC   : 0.8457
5 forward-chaining folds: [0.8316 0.8243 0.8279 0.8541 0.842 ] mean 0.836


## 6.4 Baseline, logistic regression, random forest, gradient boosting

Four contenders, from simplest to most complex.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep, mlprep
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

o, c, p = prep.load_clean(DATA)
d = mlprep.late_frame(o)
train, test = d[d.order_date < "2024-10-01"], d[d.order_date >= "2024-10-01"]
F = mlprep.CAT + mlprep.NUM
pre = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), mlprep.CAT), ("n", StandardScaler(), mlprep.NUM)])
models = {
    "logistic regression": make_pipeline(pre, LogisticRegression(max_iter=1000)),
    "random forest": make_pipeline(pre, RandomForestClassifier(200, min_samples_leaf=20, random_state=0, n_jobs=-1)),
    "gradient boosting": make_pipeline(pre, HistGradientBoostingClassifier(random_state=0)),
}
rows = [("rule: promised 45", roc_auc_score(test.late, test.promised_minutes == 45), average_precision_score(test.late, (test.promised_minutes == 45).astype(float)))]
for name, m in models.items():
    s = m.fit(train[F], train.late).predict_proba(test[F])[:, 1]
    rows.append((name, roc_auc_score(test.late, s), average_precision_score(test.late, s)))
print("base rate (PR-AUC of a random model):", round(test.late.mean(), 3))
print(pd.DataFrame(rows, columns=["model", "ROC-AUC", "PR-AUC"]).round(4).to_string(index=False))

base rate (PR-AUC of a random model): 0.142
              model  ROC-AUC  PR-AUC
  rule: promised 45   0.7392  0.2488
logistic regression   0.8438  0.4553
      random forest   0.8369  0.4530
  gradient boosting   0.8363  0.4475


## 6.5 Metrics, thresholds and money

A model outputs probabilities; a decision needs a yes or no. Choosing the **threshold** is a business decision. At any threshold, count four outcomes: true positives (flagged and late), false positives (flagged, not late), false negatives (missed late orders), true negatives. **Precision** = TP ÷ (TP + FP) tells how many flags are right; **recall** = TP ÷ (TP + FN) tells how many late orders we catch.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep, mlprep
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

o, c, p = prep.load_clean(DATA)
d = mlprep.late_frame(o)
tr = d[d.order_date < "2024-07-01"]
va = d[(d.order_date >= "2024-07-01") & (d.order_date < "2024-10-01")]
te = d[d.order_date >= "2024-10-01"]
F = mlprep.CAT + mlprep.NUM
pre = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), mlprep.CAT), ("n", StandardScaler(), mlprep.NUM)])
m = make_pipeline(pre, LogisticRegression(max_iter=1000)).fit(tr[F], tr.late)

def cost(y, flag):
    y, flag = np.asarray(y), np.asarray(flag)
    return np.where(flag, 5 + 20 * y, 60 * y).sum()

pv, pt = m.predict_proba(va[F])[:, 1], m.predict_proba(te[F])[:, 1]
ths = np.round(np.arange(0.05, 0.61, 0.01), 2)
best = ths[np.argmin([cost(va.late, pv >= t) for t in ths])]
print("best threshold on validation:", best)
for label, flag in [("flag nobody", np.zeros(len(te), bool)), ("flag everybody", np.ones(len(te), bool)), (f"model at {best}", pt >= best)]:
    tp = (flag & (te.late.values == 1)).sum()
    print(f"{label:18s} cost {cost(te.late, flag):9,.0f}  flagged {flag.sum():5d}  recall {tp / te.late.sum():.3f}  precision {tp / max(flag.sum(), 1):.3f}")

best threshold on validation: 0.11
flag nobody        cost    36,120  flagged     0  recall 0.000  precision 0.000
flag everybody     cost    33,250  flagged  4242  recall 1.000  precision 0.142
model at 0.11      cost    23,915  flagged  1719  recall 0.864  precision 0.303


## 6.6 Explaining the model

Stakeholders will not act on a black box. Three tools of increasing power.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep, mlprep
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance

o, c, p = prep.load_clean(DATA)
d = mlprep.late_frame(o)
train, test = d[d.order_date < "2024-10-01"], d[d.order_date >= "2024-10-01"]
F = mlprep.CAT + mlprep.NUM
pre = ColumnTransformer([("c", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), mlprep.CAT)], remainder="passthrough")
m = make_pipeline(pre, HistGradientBoostingClassifier(random_state=0)).fit(train[F], train.late)
r = permutation_importance(m, test[F], test.late, scoring="roc_auc", n_repeats=5, random_state=0)
imp = pd.Series(r.importances_mean, index=F).sort_values(ascending=False)
print("drop in test AUC when a feature is shuffled:")
print(imp.round(4).to_string())

# partial dependence by hand for the hour
base = test[F].copy()
curve = {}
for h in [8, 12, 15, 18, 20, 23]:
    x = base.copy(); x["order_hour"] = h; x["evening"] = int(18 <= h <= 21)
    curve[h] = round(float(m.predict_proba(x)[:, 1].mean()), 3)
print("average predicted P(late) if every order were placed at hour h:", curve)

drop in test AUC when a feature is shuffled:
promised_minutes    0.2222
city                0.0632
evening             0.0255
quantity            0.0136
category            0.0018
channel             0.0003
weekend            -0.0002
order_hour         -0.0005
discount           -0.0032
average predicted P(late) if every order were placed at hour h: {8: 0.115, 12: 0.116, 15: 0.112, 18: 0.233, 20: 0.222, 23: 0.114}


In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep, mlprep
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

o, c, p = prep.load_clean(DATA)
d = mlprep.late_frame(o).dropna(subset=["city"])
train, test = d[d.order_date < "2024-10-01"], d[d.order_date >= "2024-10-01"].copy()
F = mlprep.CAT + mlprep.NUM
pre = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), mlprep.CAT), ("n", StandardScaler(), mlprep.NUM)])
m = make_pipeline(pre, LogisticRegression(max_iter=1000)).fit(train[F], train.late)
test["flag"] = m.predict_proba(test[F])[:, 1] >= 0.11
g = test.groupby("city").apply(lambda x: pd.Series({"orders": len(x), "late_rate": x.late.mean(),
        "flagged": x.flag.mean(), "recall": (x.flag & (x.late == 1)).sum() / x.late.sum()}), include_groups=False)
g["orders"] = g.orders.astype(int)
print(g.round(3).to_string())

orders  late_rate  flagged  recall
city                                         
Bengaluru    1209      0.236    0.557   0.923
Chennai       554      0.067    0.215   0.622
Delhi         877      0.103    0.438   0.911
Mumbai       1041      0.158    0.462   0.896
Pune          539      0.045    0.072   0.167


## 6.7 An AutoML-style comparison table

AutoML tools (Power BI's automated ML in dataflows, cloud services, `auto-sklearn`) try many models and settings and rank them on cross-validation. You can reproduce the idea in twenty lines: a loop over candidates with small parameter grids, scored by forward-chaining cross-validation on the training period, then one final score on the untouched test period. The important discipline: **the test set picks nothing**.

In [ ]:
DATA = "data/"
import sys, time
sys.path.insert(0, DATA)
import prep, mlprep
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import roc_auc_score

o, c, p = prep.load_clean(DATA)
d = mlprep.late_frame(o).sort_values("order_date")
train, test = d[d.order_date < "2024-10-01"], d[d.order_date >= "2024-10-01"]
F = mlprep.CAT + mlprep.NUM
pre = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), mlprep.CAT), ("n", StandardScaler(), mlprep.NUM)])
cands = {
    "logreg C=0.1": LogisticRegression(C=0.1, max_iter=1000),
    "logreg C=1": LogisticRegression(C=1, max_iter=1000),
    "logreg C=10": LogisticRegression(C=10, max_iter=1000),
    "forest depth 6": RandomForestClassifier(200, max_depth=6, random_state=0, n_jobs=-1),
    "forest leaf 20": RandomForestClassifier(200, min_samples_leaf=20, random_state=0, n_jobs=-1),
    "boosting lr 0.05": HistGradientBoostingClassifier(learning_rate=0.05, max_iter=150, random_state=0),
    "boosting lr 0.1": HistGradientBoostingClassifier(learning_rate=0.1, max_iter=100, random_state=0),
}
rows = []
for name, est in cands.items():
    pipe = make_pipeline(pre, est)
    t = time.perf_counter()
    cv = cross_val_score(pipe, train[F], train.late, cv=TimeSeriesSplit(4), scoring="roc_auc")
    pipe.fit(train[F], train.late)
    rows.append((name, cv.mean(), cv.std(), roc_auc_score(test.late, pipe.predict_proba(test[F])[:, 1])))
tab = pd.DataFrame(rows, columns=["candidate", "cv_auc", "cv_std", "test_auc"]).sort_values("cv_auc", ascending=False)
print(tab.round(4).to_string(index=False))
print("best by CV:", tab.iloc[0].candidate)

candidate  cv_auc  cv_std  test_auc
      logreg C=1  0.8338  0.0142    0.8438
     logreg C=10  0.8337  0.0143    0.8437
    logreg C=0.1  0.8336  0.0144    0.8438
  forest depth 6  0.8315  0.0157    0.8420
  forest leaf 20  0.8299  0.0163    0.8380
boosting lr 0.05  0.8193  0.0208    0.8386
 boosting lr 0.1  0.8152  0.0230    0.8365
best by CV: logreg C=1


## 6.8 Second target: predicting churn

The same ideas apply when the unit is a customer. Define **churn** precisely (an experienced analyst spends half the project here): "Among customers who ordered in the 90 days before the cut-off date, those with no order in the 90 days after it." Features come from the past only: *recency* (days since last order), *frequency*, *monetary* value, average delivery time, return rate, segment and age. Train at one cut-off date and test at a later one, which is the production situation.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

o, c, p = prep.load_clean(DATA)

def build(cut):
    cut = pd.Timestamp(cut)
    past = o[o.order_date < cut]
    fut = o[(o.order_date >= cut) & (o.order_date < cut + pd.Timedelta(days=90))]
    g = past.groupby("customer_id")
    f = pd.DataFrame({"recency": (cut - g.order_date.max()).dt.days, "frequency": g.size(), "monetary": g.revenue.sum(),
                      "avg_delay": g.delivery_minutes.mean(),
                      "ret_rate": g.apply(lambda d: (d.status == "returned").mean(), include_groups=False),
                      "n90": past[past.order_date >= cut - pd.Timedelta(days=90)].groupby("customer_id").size()})
    f["n90"] = f.n90.fillna(0)
    f = f[f.n90 > 0].join(c.set_index("customer_id")[["segment", "age"]])
    f["avg_delay"] = f.avg_delay.fillna(f.avg_delay.median())
    f["age"] = f.age.fillna(f.age.median())
    f["premium"] = (f.segment == "Premium").astype(int)
    f["student"] = (f.segment == "Student").astype(int)
    f["churn"] = (~f.index.isin(fut.customer_id)).astype(int)
    return f.drop(columns="segment")

tr, te = build("2024-06-30"), build("2024-09-30")
X = [col for col in tr.columns if col != "churn"]
print("customers:", len(tr), "train churn rate:", round(tr.churn.mean(), 3), "| test:", len(te), round(te.churn.mean(), 3))
print("baseline rule (longer since last order = more churn) AUC:", round(roc_auc_score(te.churn, te.recency), 4))
for name, m in [("logistic regression", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
                ("gradient boosting", HistGradientBoostingClassifier(random_state=0))]:
    m.fit(tr[X], tr.churn)
    print(f"{name:20s} test AUC:", round(roc_auc_score(te.churn, m.predict_proba(te[X])[:, 1]), 4))

customers: 1880 train churn rate: 0.23 | test: 2166 0.311
baseline rule (longer since last order = more churn) AUC: 0.5851
logistic regression  test AUC: 0.7057
gradient boosting    test AUC: 0.6776


## 6.9 Process, tools and what to hand over

## Chapter recap

## Practice set

Problem 6.10.1 Why can you not use delivery_minutes to predict lateness at order time?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ways/ch06.html